# Experiment 9: Building an Agentic AI System with Tool Use Using LangGraph

**Course:** HAAI302COM - Artificial Intelligence Laboratory

## 1. Aim

To design and implement a stateful, multi-step AI agent using LangGraph that can use tools (web search, calculator, code execution) to solve tasks autonomously, with human-in-the-loop checkpoints.

## 2. Objectives

- To understand how an LLM agent decides which tool to call and when to stop.
- To define tools (calculator, Python code execution, web search) for an agent.
- To build the agent loop as a LangGraph graph with nodes, edges and state.
- To give the agent memory across turns with a checkpointer.
- To add a human approval step before a risky tool runs, and to approve or reject the action.

## 3. Software Requirements

- Python 3.9 or later
- Google Colab or Jupyter Notebook
- API key for the language model (Gemini through the OpenAI-compatible endpoint is used here)
- Internet connection (model API and web search)
- Required Python libraries: `langgraph`, `langchain-openai`, `langchain-core`, `ddgs`, `pandas`

## 4. Theory

### A. What is an agent?

A plain LLM call answers once from its memory. An **agent** repeats a loop: the model reads the conversation, decides either to call a tool or to give the final answer, the tool runs, its result is added to the conversation, and the model decides again.

`user question -> LLM -> (tool call -> tool result -> LLM)* -> final answer`

The model cannot run code or browse the web by itself; it only produces a structured request ("call `calculator` with `48271 * 9371`"), and our program executes it.

### B. Tools

| Tool | Purpose | Why the model needs it |
|---|---|---|
| Calculator | Exact arithmetic | LLMs make mistakes in long multiplication. |
| Python execution | Dates, loops, data processing | A program is more reliable than mental work. |
| Web search | Current or unknown facts | The model's knowledge is fixed at training time. |

The **description** of each tool is what the model reads to choose the right one.

### C. LangGraph

LangGraph models an agent as a graph.

| Concept | Meaning |
|---|---|
| State | The data that flows through the graph (here, the list of messages). |
| Node | A step that reads the state and returns an update (the LLM node, the tool node). |
| Edge | The connection between nodes. A conditional edge chooses the next node (tools or end). |
| Checkpointer | Saves the state after every step, which gives memory across turns and allows pausing. |

### D. Human-in-the-loop

Some actions are risky (running code, sending e-mail, deleting data). LangGraph can **interrupt** the graph before a node, save the state, and wait. A person inspects the pending tool call and approves it, or rejects it with feedback, and the graph resumes from the saved state.

## 5. Algorithm / Procedure

1. Install and import the libraries.
2. Define the three tools.
3. Create the LLM and bind the tools to it.
4. Build the graph: an agent node, a tool node and the conditional edge between them.
5. Run tasks that need different tools and record the tool calls.
6. Add a checkpointer and test memory across turns.
7. Compile the graph with an interrupt before the tool node; approve one request and reject another.
8. Compare the results and record observations.

## 6. Python Program

### Step 1: Install the required libraries

In [1]:
!pip install -q langgraph langchain-openai langchain-core ddgs pandas

### Step 2: Define the tools

The `@tool` decorator turns a Python function into a tool. The docstring becomes the description that the model reads, so it says what the tool does and when to use it.

In [2]:
import os, sys, math, time, subprocess, tempfile, warnings, pandas as pd
from dotenv import load_dotenv, find_dotenv
from langchain_core.tools import tool
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 90); pd.set_option("display.width", 180)
load_dotenv(find_dotenv(usecwd=True))

@tool
def calculator(expression: str) -> str:
    '''Evaluate an arithmetic expression exactly, for example '48271 * 9371' or 'sqrt(2) + 7 ** 2'. Use it for any calculation.'''
    try:
        return str(eval(expression, {"__builtins__": {}}, vars(math)))      # only math functions are visible
    except Exception as e:
        return f"Error: {e}"

@tool
def run_python(code: str) -> str:
    '''Run Python code and return what it prints. Use it for dates, loops, primes or data processing. The code must print its result.'''
    with tempfile.TemporaryDirectory() as folder:
        try:
            r = subprocess.run([sys.executable, "-I", "-c", code], capture_output=True, text=True, timeout=5, cwd=folder)
            return (r.stdout + r.stderr).strip()[:1500] or "(no output)"
        except subprocess.TimeoutExpired:
            return "Error: the code did not finish within 5 seconds."

@tool
def web_search(query: str) -> str:
    '''Search the web and return the top results. Use it for facts about people, companies, software or recent events.'''
    from ddgs import DDGS
    for attempt in range(3):
        try:
            hits = DDGS().text(query, max_results=3)
            return "\n".join(f"- {h['title']}: {h['body'][:250]}" for h in hits)
        except Exception:
            time.sleep(2)
    return "Search failed."

tools = [calculator, run_python, web_search]
print(calculator.invoke("48271 * 9371"), "|", run_python.invoke("print(sum(range(101)))"))

452347541 | 5050


### Step 3: Build the agent graph

`bind_tools` tells the model which tools exist. The graph has two nodes. `agent` calls the model. `tools` (a ready-made `ToolNode`) runs whatever tools the model requested. The conditional edge `tools_condition` goes to `tools` if the model asked for a tool and to the end otherwise; after the tools run, control returns to `agent`.

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

llm = ChatOpenAI(model="gemini-2.5-flash-lite", temperature=0, api_key=os.environ["GEMINI_API_KEY"],
                 base_url="https://generativelanguage.googleapis.com/v1beta/openai/", max_retries=4).bind_tools(tools)
SYSTEM = SystemMessage("You are a careful assistant. Use the calculator for arithmetic, run_python for code and date "
                       "calculations, and web_search for facts you are not sure about. After a tool returns, always "
                       "reply to the user with the final answer in one sentence.")

def agent(state: MessagesState):
    return {"messages": [llm.invoke([SYSTEM] + state["messages"])]}

def build(**compile_options):
    g = StateGraph(MessagesState)
    g.add_node("agent", agent)
    g.add_node("tools", ToolNode(tools))
    g.add_edge(START, "agent")
    g.add_conditional_edges("agent", tools_condition)     # agent -> tools, or agent -> end
    g.add_edge("tools", "agent")
    return g.compile(checkpointer=MemorySaver(), **compile_options)

app = build()
print(app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	agent(agent)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> agent;
	agent -.-> __end__;
	agent -.-> tools;
	tools --> agent;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



### Step 4: Run tasks that need different tools

For every task we record which tools the agent called and its final answer. Each task gets its own `thread_id`, which names a separate conversation in the checkpointer.

In [4]:
def run(app, question, thread):
    out = app.invoke({"messages": [HumanMessage(question)]}, {"configurable": {"thread_id": thread}})
    used = [c["name"] for m in out["messages"] if getattr(m, "tool_calls", None) for c in m.tool_calls]
    return used, out["messages"][-1].content

tasks = ["What is 48271 multiplied by 9371?",
         "How many days are there between 2024-03-15 and 2025-11-02?",
         "Who created the Linux kernel, and in which year was the first version released?",
         "A laptop costs 82500 rupees and gets a 15 percent discount, then 18 percent GST is added. What is the final price?",
         "What is the capital of France?"]
rows = []
for i, q in enumerate(tasks):
    used, answer = run(app, q, f"task-{i}")
    rows.append({"Task": q[:60], "Tools called": ", ".join(used) or "none", "Answer": answer[:110]})
results = pd.DataFrame(rows)
results

,Task,Tools called,Answer
0,What is 48271 multiplied by 9371?,calculator,"48271 multiplied by 9371 is 452,347,541."
1,How many days are there between 2024-03-15 and 2025-11-02?,run_python,There are 597 days between 2024-03-15 and 2025-11-02.
2,"Who created the Linux kernel, and in which year was the firs",web_search,"Linus Torvalds created the Linux kernel, and the first version was released in 1991."
3,"A laptop costs 82500 rupees and gets a 15 percent discount,",calculator,The final price of the laptop after the discount and GST is 82747.5 rupees.
4,What is the capital of France?,web_search,The capital of France is Paris.


### Step 5: Memory across turns

Because the graph was compiled with a checkpointer, two calls with the same `thread_id` share the conversation. A different `thread_id` starts with an empty memory.

In [5]:
cfg = {"configurable": {"thread_id": "chat-1"}}
app.invoke({"messages": [HumanMessage("My name is Aditya and I study at PCCOER.")]}, cfg)
print("same thread     :", app.invoke({"messages": [HumanMessage("Where do I study?")]}, cfg)["messages"][-1].content)
print("different thread:", app.invoke({"messages": [HumanMessage("Where do I study?")]}, {"configurable": {"thread_id": "chat-2"}})["messages"][-1].content)

same thread     : You study at PCCOER.


different thread: I cannot answer this question. I do not have access to your personal information.


### Step 6: Human-in-the-loop approval

`interrupt_before=["tools"]` makes the graph stop just before any tool runs. The saved state contains the tool call the model wants to make, which a person can inspect. Calling `invoke(None, config)` resumes and approves. To reject, we add a tool message that carries the reviewer's feedback in place of running the tool.

In [6]:
hitl = build(interrupt_before=["tools"])
Q = "Use Python to print the sum of the squares of the numbers 1 to 10."

cfg1 = {"configurable": {"thread_id": "review-1"}}
hitl.invoke({"messages": [HumanMessage(Q)]}, cfg1)
state = hitl.get_state(cfg1)
print("Paused before node:", state.next)
print("Pending tool call :", state.values["messages"][-1].tool_calls[0]["name"], state.values["messages"][-1].tool_calls[0]["args"])

Paused before node: ('tools',)
Pending tool call : run_python {'code': '\nsum_of_squares = 0\nfor i in range(1, 11):\n    sum_of_squares += i**2\nprint(sum_of_squares)\n'}


In [7]:
# APPROVE: resume with no new input and the tool runs
approved = hitl.invoke(None, cfg1)
print("Approved ->", approved["messages"][-1].content)

Approved -> The sum of the squares of the numbers from 1 to 10 is 385.


In [8]:
# REJECT: replace the tool run by a message from the reviewer
cfg2 = {"configurable": {"thread_id": "review-2"}}
hitl.invoke({"messages": [HumanMessage(Q)]}, cfg2)
call = hitl.get_state(cfg2).values["messages"][-1].tool_calls[0]
hitl.update_state(cfg2, {"messages": [ToolMessage("REJECTED by the reviewer: do not run code. Explain how to compute it by hand.",
                                                  tool_call_id=call["id"], name=call["name"])]}, as_node="tools")
rejected = hitl.invoke(None, cfg2)
print("Rejected ->", rejected["messages"][-1].content[:300])

Rejected -> I cannot execute Python code. To find the sum of the squares of the numbers from 1 to 10, you would square each number (1*1, 2*2, ..., 10*10) and then add all those squared values together.


## 7. Experimental Observations

### Observation Table 1: Tool selection for different tasks

In [9]:
results

,Task,Tools called,Answer
0,What is 48271 multiplied by 9371?,calculator,"48271 multiplied by 9371 is 452,347,541."
1,How many days are there between 2024-03-15 and 2025-11-02?,run_python,There are 597 days between 2024-03-15 and 2025-11-02.
2,"Who created the Linux kernel, and in which year was the firs",web_search,"Linus Torvalds created the Linux kernel, and the first version was released in 1991."
3,"A laptop costs 82500 rupees and gets a 15 percent discount,",calculator,The final price of the laptop after the discount and GST is 82747.5 rupees.
4,What is the capital of France?,web_search,The capital of France is Paris.


### Observation Table 2: Human-in-the-loop decisions

In [10]:
pd.DataFrame({"Decision": ["Approve", "Reject"],
              "Tool executed": ["yes (run_python)", "no"],
              "Final answer": [approved["messages"][-1].content[:100], rejected["messages"][-1].content[:100]]}).set_index("Decision")

,Tool executed,Final answer
Decision,,
Approve,yes (run_python),The sum of the squares of the numbers from 1 to 10 is 385.
Reject,no,I cannot execute Python code. To find the sum of the squares of the numbers from 1 to ...


**Observations**

- The agent chose a suitable tool for four of the five tasks: `calculator` for the multiplication (452,347,541) and for the price calculation, `run_python` for the date difference (597 days), and `web_search` for the Linux question (Linus Torvalds, 1991). The results can be checked by hand: 82,500 x 0.85 x 1.18 = 82,747.5 rupees, and 2024-03-15 to 2025-11-02 is 365 + 232 = 597 days.
- For the question "What is the capital of France?" the agent still called `web_search`, although the answer is common knowledge. The answer was correct, but the unnecessary call costs time, and tool use should be guided by the system prompt and the tool descriptions.
- Memory works through the checkpointer: with the same `thread_id` the agent remembered that the user studies at PCCOER, while a new `thread_id` had no such information and the agent said it could not answer.
- With `interrupt_before=["tools"]` the graph stopped before the tool node and exposed the pending call (`run_python` with a short loop). Approving it executed the code and the agent reported the correct sum, 385.
- Rejecting the call replaced the code run with the reviewer's message. The agent did not execute anything and explained how to compute the sum by hand instead, so the human decision changed the agent's behaviour without any change to the code of the agent itself.

## 8. Result

A stateful agent was built with LangGraph that uses three tools (calculator, Python execution and web search). It selected the right tool for the arithmetic, date and factual-search tasks and returned correct answers (452,347,541; 597 days; 82,747.5 rupees; Linus Torvalds, 1991). The checkpointer gave the agent memory within a conversation thread. A human-in-the-loop interrupt before the tool node allowed a reviewer to approve a code-execution request, which then returned 385, or to reject it, in which case the agent explained the calculation without running code.

## 9. Conclusion

An agent turns a language model into a system that can act: it chooses tools, reads their results and continues until the task is solved. LangGraph makes this loop explicit as a graph with state, nodes and conditional edges, and a checkpointer adds memory and the ability to pause. The human-in-the-loop interrupt gives control over risky actions, because a person can inspect a pending tool call and approve or reject it before anything runs. Good tool descriptions and a clear system prompt are essential for reliable behaviour.